In [ ]:
import pandas as pd

comportamiento_bancario = pd.read_csv("data/comportamiento_bancario.csv")

In [3]:
comportamiento_bancario

,ID_Usuario,Mes_Referencia,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
0,1,2024-01,12768.11,3
1,1,2024-02,15817.67,3
2,1,2024-03,13138.66,0
3,1,2024-04,19453.13,0
4,1,2024-05,14943.44,59
...,...,...,...,...
661134,28993,2025-08,7228.05,4
661135,28993,2025-09,9180.89,7
661136,28993,2025-10,9008.43,5
661137,28993,2025-11,6677.52,3


In [5]:
comportamiento_bancario["Mes_Referencia"] = pd.to_datetime(
    comportamiento_bancario["Mes_Referencia"]
)

print("Usuarios distintos:",
      comportamiento_bancario["ID_Usuario"].nunique())
print("Filas duplicadas:",
      comportamiento_bancario.duplicated().sum())
print(
    "Duplicados de usuario y mes:",
    comportamiento_bancario.duplicated(
        subset=["ID_Usuario", "Mes_Referencia"]
    ).sum()
)
print(
    "Período:",
    comportamiento_bancario["Mes_Referencia"].min().date(),
    "a",
    comportamiento_bancario["Mes_Referencia"].max().date()
)

meses_por_usuario = comportamiento_bancario.groupby(
    "ID_Usuario"
)["Mes_Referencia"].nunique()

print("\nResumen de meses disponibles por usuario:")
display(meses_por_usuario.describe().round(2))

Usuarios distintos: 28993
Filas duplicadas: 0
Duplicados de usuario y mes: 0
Período: 2024-01-01 a 2025-12-01

Resumen de meses disponibles por usuario:


count    28993.00
mean        22.80
std          1.07
min         17.00
25%         22.00
50%         23.00
75%         24.00
max         24.00
Name: Mes_Referencia, dtype: float64

In [6]:
banco = comportamiento_bancario.copy()
banco["Mes_Referencia"] = pd.to_datetime(banco["Mes_Referencia"])
banco = banco.sort_values(["ID_Usuario", "Mes_Referencia"])

col_saldo = "Saldo_Promedio_Mensual"

resumen_saldo = banco.groupby("ID_Usuario").agg(
    saldo_promedio_historico=(col_saldo, "mean"),
    saldo_inicial=(col_saldo, "first"),
    saldo_final=(col_saldo, "last")
)

for cantidad_meses in [3, 6]:
    promedios_recientes = (
        banco.groupby("ID_Usuario")
        .tail(cantidad_meses)
        .groupby("ID_Usuario")[col_saldo]
        .mean()
    )
    resumen_saldo[f"promedio_ultimos_{cantidad_meses}_meses"] = promedios_recientes

resumen_saldo["cambio_inicio_fin"] = (
    resumen_saldo["saldo_final"] - resumen_saldo["saldo_inicial"]
)

display(resumen_saldo.head())

,saldo_promedio_historico,saldo_inicial,saldo_final,promedio_ultimos_3_meses,promedio_ultimos_6_meses,cambio_inicio_fin
ID_Usuario,,,,,,
1,20017.683913,12768.11,23343.81,24550.633333,24299.358333,10575.70
2,3781.878571,3335.41,4576.35,4515.880000,4597.363333,1240.94
3,4577.649545,4683.17,4133.35,4556.326667,4890.723333,-549.82
4,4992.794583,3971.51,6122.52,5571.863333,5283.425000,2151.01
5,2556.866667,3738.41,1753.87,1875.490000,1948.161667,-1984.54


In [6]:
banco = comportamiento_bancario.copy()
banco["Mes_Referencia"] = pd.to_datetime(banco["Mes_Referencia"])
banco = banco.sort_values(["ID_Usuario", "Mes_Referencia"])

col_saldo = "Saldo_Promedio_Mensual"

resumen_saldo = banco.groupby("ID_Usuario").agg(
    saldo_promedio_historico=(col_saldo, "mean"),
    saldo_inicial=(col_saldo, "first"),
    saldo_final=(col_saldo, "last")
)

for cantidad_meses in [3, 6]:
    promedios_recientes = (
        banco.groupby("ID_Usuario")
        .tail(cantidad_meses)
        .groupby("ID_Usuario")[col_saldo]
        .mean()
    )
    resumen_saldo[f"promedio_ultimos_{cantidad_meses}_meses"] = promedios_recientes

resumen_saldo["cambio_inicio_fin"] = (
    resumen_saldo["saldo_final"] - resumen_saldo["saldo_inicial"]
)

display(resumen_saldo.head())

,saldo_promedio_historico,saldo_inicial,saldo_final,promedio_ultimos_3_meses,promedio_ultimos_6_meses,cambio_inicio_fin
ID_Usuario,,,,,,
1,20017.683913,12768.11,23343.81,24550.633333,24299.358333,10575.70
2,3781.878571,3335.41,4576.35,4515.880000,4597.363333,1240.94
3,4577.649545,4683.17,4133.35,4556.326667,4890.723333,-549.82
4,4992.794583,3971.51,6122.52,5571.863333,5283.425000,2151.01
5,2556.866667,3738.41,1753.87,1875.490000,1948.161667,-1984.54


Para ver como se multiplican con el merge directo (MAL)

In [9]:
merge_directo = solicitudes_train.merge(
    comportamiento_bancario,
    on="ID_Usuario",
    how="left"
)

print("Filas de solicitudes:", len(solicitudes_train))
print("Filas del merge directo:", len(merge_directo))

Filas de solicitudes: 32416
Filas del merge directo: 739137


Filtrar por fecha, resumir y volver a unir (BIEEEEN)

In [10]:
solicitudes = solicitudes_train.copy()
banco = comportamiento_bancario.copy()

solicitudes["Fecha_Solicitud"] = pd.to_datetime(solicitudes["Fecha_Solicitud"])
banco["Mes_Referencia"] = pd.to_datetime(banco["Mes_Referencia"])

historial = solicitudes[
    ["ID_Solicitud", "ID_Usuario", "Fecha_Solicitud"]
].merge(
    banco,
    on="ID_Usuario",
    how="left"
)

# Conservamos solo los meses anteriores a cada solicitud
historial = historial[
    historial["Mes_Referencia"] < historial["Fecha_Solicitud"]
]

# Resumimos el historial: una fila por solicitud
resumen = historial.groupby("ID_Solicitud").agg(
    meses_historial=("Mes_Referencia", "count"),
    saldo_promedio=("Saldo_Promedio_Mensual", "mean"),
    atraso_promedio=("Dias_Atraso_Otras_Deudas", "mean")
).reset_index()

# Agregamos el resumen a las solicitudes
datos = solicitudes.merge(
    resumen,
    on="ID_Solicitud",
    how="left",
    validate="one_to_one"
)

print("Filas de solicitudes:", len(solicitudes))
print("Filas del dataset final:", len(datos))

Filas de solicitudes: 32416
Filas del dataset final: 32416


In [11]:
datos

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud,Estado_Final,Mes_Solicitud,meses_historial,saldo_promedio,atraso_promedio
0,1,12698,7900,0.2333,4461,Consumo,2025-04-01,1,2025-04,14,3695.225714,4.857143
1,2,20744,17400,0.2507,3889,Vivienda,2025-12-01,1,2025-12,21,5976.456667,8.809524
2,3,6830,8600,0.2368,4035,Vehiculo,2025-06-01,0,2025-06,15,178.574667,20.733333
3,4,5722,13300,0.2826,4018,Vehiculo,2025-07-01,1,2025-07,18,7589.356667,0.944444
4,5,24719,6600,0.2314,3357,Otros,2025-01-01,1,2025-01,12,3888.616667,3.750000
...,...,...,...,...,...,...,...,...,...,...,...,...
32411,32412,22337,38600,0.2640,7654,Vivienda,2025-04-01,1,2025-04,13,10926.146154,4.846154
32412,32413,15817,4700,0.2872,1388,Otros,2025-10-01,0,2025-10,19,504.819474,19.421053
32413,32414,330,5000,0.2699,2445,Vehiculo,2025-05-01,1,2025-05,16,2730.808125,10.125000
32414,32415,2907,28600,0.2907,8412,Negocio,2025-02-01,1,2025-02,12,15493.373333,4.666667
